# Auto-encodeurs simples avec Keras

## Import de TensorFlow et des autres librairies nécessaires

In [ ]:
import keras
import matplotlib.pyplot as plt
import numpy
import scipy.interpolate
import tensorflow as tf

from keras import layers

## Chargement de MNIST

Nous allons utiliser un prétraîtement légèrement différent des autres fois : étant donné que nous voulons pouvoir prédire les valeurs données en entrée en sortie (principe de l'auto-encodage), nous allons simplement projeter ces valeurs dans $[0, 1]$ au lieu de $[0, 255]$. Notez qu'habituellement nous ne faisons pas ça : nous normalisons en centrant sur zéro et en divisant par l'écart-type.

In [ ]:
(X_train, y_train), (X_test, y_test) = keras.datasets.mnist.load_data()
nb_classes = 10
input_dim = 28 * 28
X_train = X_train.reshape(-1, input_dim) / 255.0
X_test = X_test.reshape(-1, input_dim) / 255.0

In [ ]:
X_train.shape

## Création de l'autoencodeur

Vous devriez être capable de créer le modèle d'autoencoder de base par vous-même.

Attention aux choix des fonctions d'activations et loss !


In [ ]:
# Votre code ici
encoding_dim = 20
encoder = keras.Sequential()
decoder = keras.Sequential()
autoencoder = keras.Sequential()

### Solution

In [ ]:
encoding_dim = 20
encoder = keras.Sequential(name="encoder")
encoder.add(layers.Input(shape=(input_dim,)))
encoder.add(layers.Dense(encoding_dim, activation="relu"))

decoder = keras.Sequential(name="decoder")
decoder.add(layers.Input(shape=(encoding_dim,)))
decoder.add(layers.Dense(input_dim, activation="sigmoid"))

autoencoder = keras.Sequential(name="autoencoder")
autoencoder.add(encoder)
autoencoder.add(decoder)
autoencoder.summary()

autoencoder.compile(optimizer="adam", loss="mean_squared_error")

In [ ]:
encoding_dim = 20
dense_params = dict(activation="relu", kernel_initializer="orthogonal")

encoder = keras.Sequential(name="encoder")
encoder.add(layers.Input(shape=(input_dim,)))
encoder.add(layers.Dense(150, **dense_params))
encoder.add(layers.Dense(150, **dense_params))
encoder.add(layers.Dense(150, **dense_params))
encoder.add(layers.Dense(150, **dense_params))
encoder.add(layers.Dense(50, **dense_params))
encoder.add(layers.Dense(encoding_dim, kernel_initializer="orthogonal", activation="linear"))

decoder = keras.Sequential(name="decoder")
decoder.add(layers.Input(shape=(encoding_dim,)))
decoder.add(layers.Dense(50, **dense_params))
decoder.add(layers.Dense(150, **dense_params))
decoder.add(layers.Dense(150, **dense_params))
decoder.add(layers.Dense(150, **dense_params))
decoder.add(layers.Dense(150, **dense_params))
decoder.add(layers.Dense(input_dim, activation="sigmoid", kernel_initializer="orthogonal"))

autoencoder = keras.Sequential(name="autoencoder")
autoencoder.add(encoder)
autoencoder.add(decoder)

autoencoder.summary(expand_nested=True)

autoencoder.compile(optimizer="adam", loss="mean_squared_error")

## Apprentissage

*Écrivez la ligne correspondant à l'apprentissage de votre autoencodeur :*

- *50 itérations devraient suffire*
- *Utilisez un batch de 256*

In [ ]:
# Votre code ici

### Solution

In [ ]:
autoencoder.fit(X_train, X_train,
                epochs=50,
                batch_size=256,
                validation_split=0.2)

## Prédiction sur du bruit blanc

Pour une première utilisation du décodeur, on peut regarder ce qu'il prédit sur du bruit blanc en entrée.

*Parfois, l'image générée est quasiment constante malgré le bruit aléatoire donné en entrée. À quoi cela est-il dû ?*

In [ ]:
white_noise = numpy.random.random_sample((1, encoding_dim))
plt.imshow(decoder.predict(white_noise).reshape(28, 28), cmap="gray_r")
plt.show()

## Encodage des données de test

On peut aussi, grâce à l'encodeur récupéré, encoder nos données de test vers l'espace de dimension `encoding_dim` appris.

In [ ]:
codes = encoder.predict(X_test)

In [ ]:
codes.shape

## Calcul des centroïdes de chaque chiffre dans l'espace de codage

Un autre point intéressant est de regarder en moyenne où atterrissent les exemples d'un label donné dans l'espace de codage.

In [ ]:
means = numpy.vstack([codes[y_test == i].mean(axis=0)
                      for i in range(nb_classes)])
stds = numpy.vstack([codes[y_test == i].std(axis=0)
                     for i in range(nb_classes)])

# Étude rapide des codages en test
for i in range(10):
  dimension_stats = [f"{mean:5.2f}±{std:.2f}"
                     for mean, std in zip(means[i], stds[i])]
  print(f"Classe {i} {', '.join(dimension_stats)}")

On peut à partir de là vérifier que chaque centroïde est bien décodé par quelque chose de vraisemblable par le décodeur.

In [ ]:
f, ax = plt.subplots(1, nb_classes, figsize=(1.4 * nb_classes, 2))

centroid_images = decoder.predict(means).reshape(-1, 28, 28)

for i, centroid_image in enumerate(centroid_images):
  ax[i].imshow(centroid_image, cmap="gray_r")
  ax[i].axis("off")
plt.show()

## Parcours de l'espace latent entre deux centroïdes

Maintenant que l'on sait où se trouvent les centroïdes pour un chiffre donné dans l'espace latent, on peut parcourir l'espace entre deux centroïdes pour mieux comprendre comment l'espace latent est structuré.

In [ ]:
def latent_walk(start: int, end: int, n: int = 15):
  interpolator = scipy.interpolate.interp1d([0, n - 1],
                                            means[[start, end], :],
                                            axis=0)
  interpolated_codes = interpolator(range(n))
  interpolated_images = decoder.predict(interpolated_codes).reshape(-1, 28, 28)

  f, ax = plt.subplots(1, n, figsize=(n * 1.4, 2))
  for i, interpolated_image in enumerate(interpolated_images):
    ax[i].imshow(interpolated_image, cmap="gray_r")
    ax[i].axis("off")
  plt.show()


latent_walk(7, 6)

## Auto-encodage de toute la base de test

Auto-encodez les images de train et de test et stockez les images obtenues dans les variables `X_train_pred` et `X_test_pred`

In [ ]:
# Votre code ici
X_train_pred = X_train
X_test_pred = X_test

### Solution

In [ ]:
X_train_pred = autoencoder.predict(X_train)
X_test_pred = autoencoder.predict(X_test)

## Évaluation visuelle de la performance

In [ ]:
n = 15  # Nombre de chiffres que nous allons afficher

random_indexes = numpy.random.choice(X_test_pred.shape[0],
                                     size=n,
                                     replace=False)


def display_samples(indices: numpy.ndarray,
                    X: numpy.ndarray,
                    X_pred: numpy.ndarray,
                    y: numpy.ndarray
                   ) -> None:
  _, ax = plt.subplots(2, n, figsize=(n * 1.4, 4))
  for i, index in enumerate(indices):
      # L'original en haut
      ax[0, i].set_title(y[index])
      ax[0, i].imshow(X[index].reshape(28, 28), cmap="gray_r")
      ax[0, i].axis("off")

      # La reconstruction en bas
      ax[1, i].imshow(X_pred[index].reshape(28, 28), cmap="gray_r")
      ax[1, i].axis("off")
  plt.show()


display_samples(random_indexes, X_test, X_test_pred, y_test)

## Détection d'anomalies

Il est possible de détecter des anomalies dans les données en utilisant l'erreur de reconstruction : une grande erreur induit que l'exemple est hors de la distribution normale des données.

In [ ]:
mse = keras.losses.MeanSquaredError(reduction="none")
test_anomalies = (-mse(X_test_pred, X_test).numpy()).argsort()
train_anomalies = (-mse(X_train_pred, X_train).numpy()).argsort()

n = 20

print("Pires reconstructions sur le train")
display_samples(train_anomalies[:n], X_train, X_train_pred, y_train)

print("Pires reconstructions sur le test")
display_samples(test_anomalies[:n], X_test, X_test_pred, y_test)

print("Meilleures reconstructions sur le train")
display_samples(train_anomalies[-n:], X_train, X_train_pred, y_train)

print("Meilleures reconstructions sur le test")
display_samples(test_anomalies[-n:], X_test, X_test_pred, y_test)

## À vous de jouer !

Trouvez une valeur idéale pour la taille de l'encodage ainsi qu'un modèle adapté afin que votre autoencodeur ait une perte de compression acceptable.

Quelle est votre taux de compression ?